In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_json("../eprom.json")
df.head()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0039-0001,Fall Launch 2020,percentage,10.0,2020-08-30,2020-10-01,NaN,all_channels,0,150000
1,PROMO-0029-0002,Fall Launch 2018,percentage,10.0,2018-08-30,2018-10-01,NaN,email,0,0
2,PROMO-0015-0003,Urban Blowout 2015,fixed,50.0,2015-07-30,2015-09-02,Streetwear,online,0,200000
3,PROMO-0043-0004,Fall Launch 2021,percentage,10.0,2021-08-30,2021-10-02,NaN,email,0,0
4,PROMO-0008-0005,Mid-Year Sale 2014,percentage,18.0,2014-06-23,2014-07-22,NaN,social_media,0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   promo_id             1000 non-null   str   
 1   promo_name           999 non-null    str   
 2   promo_type           1000 non-null   str   
 3   discount_value       1000 non-null   object
 4   start_date           1000 non-null   str   
 5   end_date             1000 non-null   str   
 6   applicable_category  217 non-null    str   
 7   promo_channel        1000 non-null   str   
 8   stackable_flag       1000 non-null   int64 
 9   min_order_value      1000 non-null   int64 
dtypes: int64(2), object(1), str(7)
memory usage: 78.3+ KB


In [4]:
# Kiểm tra thử promo_id có giá trị null và trống không
df[df["promo_id"].isnull() | (df["promo_id"] == "")]

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value


In [5]:
# Kiểm tra promo_name có giá trị null và trong không
df[df["promo_name"].isnull() | (df["promo_name"] == "")]

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
17,PROMO-0002-0018,NaN,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
286,PROMO-0009-0287,,percentage,10.0,2014-08-30,2014-10-01,NaN,all_channels,0,100000


In [6]:
# Quyết định không bỏ các giá trị null và trống trong promo_name và cho nó tên là  Unknown Promo
df["promo_name"] = df["promo_name"].replace("", "Unknown Promo")
df["promo_name"] = df["promo_name"].replace(np.nan, "Unknown Promo")

# Kiểm tra lại
df[df["promo_name"] == 'Unknown Promo']

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
17,PROMO-0002-0018,Unknown Promo,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
286,PROMO-0009-0287,Unknown Promo,percentage,10.0,2014-08-30,2014-10-01,NaN,all_channels,0,100000


In [7]:
# Xem các giá trị của promo_type
df["promo_type"].value_counts()

promo_type
percentage    885
fixed         115
Name: count, dtype: int64

In [8]:
# Xem các giá trị của discount_value
df["discount_value"].value_counts()

discount_value
10.0              206
18.0              206
12.0              192
20.0              175
50.0              115
15.0              102
twenty percent      1
150                 1
30%%                1
-10                 1
Name: count, dtype: int64

In [9]:
# Xem các giá trị discount_value không hợp lệ
df[
    df["discount_value"].astype("string").isin(
        ["twenty percent", "30%%", "150", "-10"]
    )
]

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
73,PROMO-0034-0074,Year-End Sale 2019,percentage,twenty percent,2019-11-18,2020-01-02,NaN,all_channels,0,50000
121,PROMO-0034-0122,Year-End Sale 2019,percentage,150,2019-11-18,2020-01-02,NaN,all_channels,0,50000
514,PROMO-0019-0515,Fall Launch 2016,percentage,30%%,2016-08-30,2016-10-01,NaN,online,0,0
622,PROMO-0012-0623,Mid-Year Sale 2015,percentage,-10,2015-06-23,2015-07-22,NaN,social_media,0,0


In [10]:
# Sửa các giá trị dạng chữ
df["discount_value"] = df["discount_value"].replace({
    "twenty percent": 20,
    "30%%": 30
})


df["discount_value"] = pd.to_numeric(
    df["discount_value"], errors="coerce"
)

# Giả định -10 bị nhập nhầm dấu
df.loc[
    (df["promo_type"] == "percentage")
    & (df["discount_value"] == -10),
    "discount_value"
] = 10

# Giả định 150 là fixed chứ không phải percentage
df.loc[121, "promo_type"] = "fixed"

df[df["promo_type"] == "fixed"]['discount_value'].value_counts()



discount_value
50.0     115
150.0      1
Name: count, dtype: int64

In [11]:
# Kiểm tra các giá trị ngày tháng có hợp lệ không
start_check = pd.to_datetime(
    df["start_date"], format="%Y-%m-%d", errors="coerce"
)

end_check = pd.to_datetime(
    df["end_date"], format="%Y-%m-%d", errors="coerce"
)

display(df[start_check.isna() | end_check.isna()])

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value


In [12]:
df["start_date"] = start_check
df["end_date"] = end_check

In [13]:
# Kiểm tra thử ngày băt đầu có sau ngày kết thúc không
df[df["start_date"] > df["end_date"]]

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
333,PROMO-0022-0334,Mid-Year Sale 2017,percentage,18.0,2026-12-31,2026-01-01,NaN,online,0,150000


In [14]:
# Loại dòng có ngày bắt đầu sau ngày kết thúc
df = df[~(df["start_date"] > df["end_date"])].copy()

# Kiểm tra lại
df[df["start_date"] > df["end_date"]]

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value


In [15]:
df["applicable_category"].value_counts(dropna=False)

applicable_category
NaN           782
Streetwear    115
Outdoor       102
Name: count, dtype: int64

In [16]:
# Giả sử các giá trị null áp dụng cho tất cả các category, nên sẽ điền giá trị All
df["applicable_category"] = df["applicable_category"].astype("category")

df["applicable_category"] = (
    df["applicable_category"]
    .str.strip()
    .replace("", pd.NA)
    .fillna("All")
)

# Kiểm tra lại
df["applicable_category"].value_counts(dropna=False)

applicable_category
All           782
Streetwear    115
Outdoor       102
Name: count, dtype: int64

In [17]:
# Xem các giá trị của promo_channel
df["promo_channel"].value_counts(dropna=False)

promo_channel
all_channels    367
online          268
email           134
social_media    127
in_store        102
N/A               1
Name: count, dtype: int64

In [18]:
# Xoá dòng này
df.loc[df["promo_channel"] == "N/A", "promo_channel"] = pd.NA
df.dropna(subset=["promo_channel"], inplace=True)
df["promo_channel"].value_counts(dropna=False)

promo_channel
all_channels    367
online          268
email           134
social_media    127
in_store        102
Name: count, dtype: int64

In [19]:
# Kiểm tra các giá trị của stackable_flag
df['stackable_flag'].value_counts(dropna=False)

stackable_flag
0    770
1    228
Name: count, dtype: int64

In [20]:
# Kiểm tra các giá trị của min_order_value
df['min_order_value'].value_counts(dropna=False)

min_order_value
0         615
150000    173
100000     93
50000      77
200000     40
Name: count, dtype: int64

In [21]:
df.info()
df.to_csv("../SilverData/eprom.csv", index=False)

<class 'pandas.DataFrame'>
Index: 998 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             998 non-null    str           
 1   promo_name           998 non-null    str           
 2   promo_type           998 non-null    str           
 3   discount_value       998 non-null    float64       
 4   start_date           998 non-null    datetime64[us]
 5   end_date             998 non-null    datetime64[us]
 6   applicable_category  998 non-null    object        
 7   promo_channel        998 non-null    str           
 8   stackable_flag       998 non-null    int64         
 9   min_order_value      998 non-null    int64         
dtypes: datetime64[us](2), float64(1), int64(2), object(1), str(4)
memory usage: 85.8+ KB
